In [1]:
import os

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
project_root = '/content/drive/MyDrive/FraudDetection_project/'
api_path = project_root + 'api/'
os.makedirs(api_path, exist_ok=True)

print("Writing requirements.txt...")

requirements = """
fastapi==0.110.0
uvicorn==0.29.0
pydantic==2.6.4
pandas==2.2.1
xgboost==2.0.3
joblib==1.3.2
scikit-learn==1.4.1.post1
"""

with open(api_path + 'requirements.txt', 'w') as f:
    f.write(requirements.strip())

print("requirements.txt saved!")

Mounted at /content/drive
Writing requirements.txt...
requirements.txt saved!


In [2]:
print("Writing main.py...")

# Notice we changed the path to point locally inside the container
api_code = """
import pandas as pd
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Global Fraud Detection Engine", version="1.0")

# Load model from the container's local directory
xgb_paysim = joblib.load('xgb_paysim_model.joblib')

class PaySimTransaction(BaseModel):
    amount: float
    balance_drop_ratio: float
    txn_velocity: int
    is_transfer_or_cashout: int
    balance_drained: int
    receiver_balance_unchanged: int

@app.get("/")
def health_check():
    return {"status": "Online", "engine": "Fraud Detection API V1"}

@app.post("/predict/paysim")
def predict_paysim(transaction: PaySimTransaction):
    try:
        input_data = pd.DataFrame([transaction.model_dump()])

        expected_features = xgb_paysim.feature_names_in_
        input_data = input_data[expected_features]

        fraud_prob = float(xgb_paysim.predict_proba(input_data)[0][1])
        is_fraud = bool(fraud_prob >= 0.50)

        if fraud_prob > 0.85:
            risk_tier = "CRITICAL - FREEZE ACCOUNT"
        elif fraud_prob > 0.50:
            risk_tier = "HIGH - BLOCK TRANSACTION"
        elif fraud_prob > 0.20:
            risk_tier = "ELEVATED - REQUIRE 2FA/OTP"
        else:
            risk_tier = "LOW - APPROVE"

        return {
            "transaction_authorized": not is_fraud,
            "fraud_probability": round(fraud_prob, 4),
            "risk_tier": risk_tier,
            "model_version": "xgb_paysim_tuned_v1"
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
"""

with open(api_path + 'main.py', 'w') as f:
    f.write(api_code.strip())

print("main.py saved!")

Writing main.py...
main.py saved!


In [3]:
import shutil

print("Writing Dockerfile and assembling the final deployment package...")

# 1. Copy the tuned model from your vault into the API folder
models_path = project_root + 'saved_models/'
shutil.copy(models_path + 'xgb_paysim_model.joblib', api_path + 'xgb_paysim_model.joblib')

# 2. Write the Dockerfile
dockerfile_content = """
# Step 1: Use an official, lightweight Python runtime as a parent image
FROM python:3.11-slim

# Step 2: Set the working directory inside the container
WORKDIR /app

# Step 3: Copy the requirements file into the container
COPY requirements.txt .

# Step 4: Install the required libraries
RUN pip install --no-cache-dir -r requirements.txt

# Step 5: Copy the API code and the machine learning model into the container
COPY main.py .
COPY xgb_paysim_model.joblib .

# Step 6: Expose the port FastAPI runs on
EXPOSE 8000

# Step 7: Define the command to run the web server when the container starts
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open(api_path + 'Dockerfile', 'w') as f:
    f.write(dockerfile_content.strip())

print("Dockerfile saved!")
print("\nDeployment Package is ready! Look inside your Drive -> FraudDetection_project -> api")

Writing Dockerfile and assembling the final deployment package...
Dockerfile saved!

Deployment Package is ready! Look inside your Drive -> FraudDetection_project -> api
